# Result-region K/V swap on the twelve-carrier message, Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16, greedy decoding. Do not switch GPUs.

Do **not** regenerate A. Do **not** overwrite `results_stage_plan4_recon12` or `results_stage_plan4_query12`. No adapter.

Base cache: generic support + transferred R. Only the result-region tensors change:
1. **tk_tv**: transferred K + transferred V (must equal generic_r exactly)
2. **fk_tv**: fresh K + transferred V
3. **tk_fv**: transferred K + fresh V
4. **fk_fv**: fresh K + fresh V (must equal generic_fresh_r exactly)

Fresh R = the same result tokens forwarded after the identical support. Keys are copied as stored; RoPE is not applied again.

**Cell 4** (`--phase check`) builds all four caches per case and verifies: support/prefix/separator tensors identical, result keys and values from the intended source, equal token IDs, cache length, masks and cursor, and conditions 1 and 4 bit-identical to their references.

**Cell 5** (`--phase compare`) runs the primary twelve-line protocol (36 cases) and the secondary single-carrier questions (432), reusing the same caches. The summary reports whether conditions 1 and 4 reproduce the recon12 / query12 outputs exactly.

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`; from `results_stage_plan4_recon12/`: `a_artifacts_12.jsonl`, `evaluations.jsonl`, `check_summary.json`; from `results_stage_plan4_query12/`: `evaluations.jsonl`.

Download `/kaggle/working/results_stage_plan4_kvmix12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_kvmix12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_kvmix12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_kvmix12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_kvmix12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
for folder, names in (("results_stage_plan4_recon12", ("a_artifacts_12.jsonl", "evaluations.jsonl", "check_summary.json")),
                      ("results_stage_plan4_query12", ("evaluations.jsonl",))):
    for name in names:
        print(folder, name, (REPO / folder / name).is_file())

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_kvmix12. Check first, then compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_kvmix12", "--split", "plan4_timing12", "--phase", "check"])
print("kvmix12 check exit", rc)
path = WORK / "results_stage_plan4_kvmix12" / "check_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_kvmix12" / "run" / "check_summary.json"
check = json.loads(path.read_text(encoding="utf-8"))
print(json.dumps({"n": check.get("n"), "all_passed": check.get("all_passed")}, indent=2))
for row in check.get("rows", []):
    bad = {c: r for c, r in row["conditions"].items() if not r["ok"]}
    if bad or not row["ok"]:
        print(row["a_state_key"], row["construction"], bad)
if not check.get("all_passed"):
    raise RuntimeError("Mixed-cache check failed. Do not run compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_kvmix12", "--split", "plan4_timing12", "--phase", "compare"])
print("kvmix12 compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_kvmix12"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_kvmix12" / "run"
data = json.loads((src / "summary.json").read_text(encoding="utf-8"))
print("twelve-line rows", data["n_twelve_rows"], "of", data["n_twelve_expected"],
      "| single-label rows", data["n_query_rows"], "of", data["n_query_expected"])
print("\nPRIMARY twelve-line labels")
for cond, block in data["twelve_line"]["labels"].items():
    print(cond, block)
print("\nfinal choice (separate)")
for cond, block in data["twelve_line"]["final_choice"].items():
    print(cond, block)
keys = ("n", "parse_ok", "exact_raw", "truncated", "correct", "ineligible_marked_eligible", "eligible_marked_ineligible")
print("\nSECONDARY single-label")
for cond, block in data["single_label"]["labels"].items():
    print(cond, {k: block.get(k) for k in keys})
print("\nreproduction of conditions 1 and 4")
print(json.dumps(data["reproduction"], indent=2))
print("\npaired by case, twelve-line labels")
for name, block in data["twelve_line"]["paired_by_case"].items():
    c = block["labels_correct"]
    print(name, {k: c.get(k) for k in ("total_delta", "mean_delta_per_case", "ci95_per_case", "cases_increased", "cases_decreased")})